# Permutation Test — HuBERT Layer 12

**Why a separate notebook:** Day 3's layer-sweep notebook (`01_features.ipynb` section 8 + `02_train_evaluate.ipynb` section 13) already ran and its version was saved — re-opening it to bolt on a permutation test would mean re-running the whole 13-layer sweep again just to get to one new cell. This notebook is standalone and does only one thing: confirm whether HuBERT layer 12's result (AUC 0.611, 95% CI [0.551, 0.683] — the current leading candidate, see `PROJECT_PLAN.md`) is distinguishable from what a model with **no real ability** could achieve on data shaped like ours, just by chance.

**What a permutation test is, briefly** (see the full explanation in conversation): shuffle the fatigue labels so any true connection between voice and fatigue is destroyed, but keep everything else about the data's structure intact — same participants, same per-participant class balance, same session grouping. Rerun the exact evaluation pipeline on this scrambled version. Do that many times to build up a picture of "what AUC looks like when there's genuinely nothing to find." Then check where the real AUC (0.611) falls against that pile of fake results — if scrambled labels rarely if ever reach 0.611, that's evidence the real result reflects something real, not noise.

**Inputs needed (Kaggle "Add Input"):**
1. The **`Fatigue_data`** dataset — for `labels.csv`.
2. **Day 3's `01_features.ipynb`** notebook output — for `hubert_base_all_layers.parquet` (only layer 12's columns are used here).

## 0. Setup

In [ ]:
import warnings
from pathlib import Path

import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
from tqdm.auto import tqdm

from sklearn.base import clone
from sklearn.exceptions import ConvergenceWarning
from sklearn.linear_model import LogisticRegression
from sklearn.model_selection import GroupKFold
from sklearn.metrics import roc_auc_score, balanced_accuracy_score
from sklearn.pipeline import Pipeline
from sklearn.preprocessing import StandardScaler
from sklearn.svm import LinearSVC

warnings.filterwarnings("ignore", category=ConvergenceWarning)

RANDOM_SEED = 42

LABELS_CSV = next(Path("/kaggle/input").rglob("labels.csv"))
HUBERT_LAYERS_PARQUET = next(Path("/kaggle/input").rglob("hubert_base_all_layers.parquet"))

print(f"LABELS_CSV            = {LABELS_CSV}")
print(f"HUBERT_LAYERS_PARQUET = {HUBERT_LAYERS_PARQUET}")

In [ ]:
labels = pd.read_csv(LABELS_CSV)
hubert_layers = pd.read_parquet(HUBERT_LAYERS_PARQUET)

df = labels.merge(hubert_layers, on="filename", how="inner")
assert len(df) == len(labels) == 280, f"expected 280 rows after merge, got {len(df)}"

LABEL_MAP = {"non_fatigued": 0, "fatigued": 1}
df["y"] = df["label"].map(LABEL_MAP)
assert df["y"].isna().sum() == 0, "unmapped label value found"

LAYER = 12
layer_cols = [c for c in df.columns if c.startswith(f"layer{LAYER}_emb_")]
assert len(layer_cols) == 1536, f"expected 1536 layer-{LAYER} columns, got {len(layer_cols)}"

X = df[layer_cols].to_numpy()
y = df["y"].to_numpy()
groups = df["participant_id"].to_numpy()

print(f"X: {X.shape} (HuBERT layer {LAYER} only), y: {y.shape}, participants: {len(np.unique(groups))}")

## 1. Candidates & Nested CV (duplicated from `02_train_evaluate.ipynb`)

Same 10 candidates, same outer-LOSO -> inner-`GroupKFold(6)` -> per-fold inner-CV threshold algorithm, same leak-free asserts used everywhere else in this project. Copied here rather than imported since this notebook runs standalone.

In [ ]:
CANDIDATE_CS = [0.001, 0.01, 0.1, 1.0, 10.0]


def build_pipeline(model_name: str, c: float) -> Pipeline:
    if model_name == "logreg":
        clf = LogisticRegression(C=c, max_iter=5000, class_weight="balanced")
    elif model_name == "linsvc":
        clf = LinearSVC(C=c, max_iter=10000, class_weight="balanced")
    else:
        raise ValueError(model_name)
    return Pipeline([("scaler", StandardScaler()), ("clf", clf)])


def make_candidates():
    return [
        (name, c, build_pipeline(name, c))
        for name in ["logreg", "linsvc"]
        for c in CANDIDATE_CS
    ]


def score_fn(fitted_pipeline: Pipeline, X_in: np.ndarray) -> np.ndarray:
    clf = fitted_pipeline.named_steps["clf"]
    if hasattr(clf, "predict_proba"):
        return fitted_pipeline.predict_proba(X_in)[:, 1]
    return fitted_pipeline.decision_function(X_in)


def best_threshold(scores: np.ndarray, y_true: np.ndarray) -> float:
    best_t, best_ba = 0.5, -1.0
    for t in np.unique(scores):
        ba = balanced_accuracy_score(y_true, (scores >= t).astype(int))
        if ba > best_ba:
            best_ba, best_t = ba, t
    return best_t

In [ ]:
def nested_loso_cv(X, y, groups, n_inner_splits=6):
    participants = np.unique(groups)
    oof_scores = np.zeros(len(y), dtype=float)
    oof_preds = np.zeros(len(y), dtype=int)
    fold_records = []
    n_outer_checked, n_inner_checked = 0, 0

    for test_participant in participants:
        outer_test_mask = groups == test_participant
        outer_train_mask = ~outer_test_mask
        assert not (set(groups[outer_train_mask]) & set(groups[outer_test_mask])), \
            f"outer leakage: {test_participant} present in both outer train and test"
        n_outer_checked += 1

        X_tr, y_tr, g_tr = X[outer_train_mask], y[outer_train_mask], groups[outer_train_mask]
        X_te = X[outer_test_mask]

        candidates = make_candidates()
        cand_aucs = {i: [] for i in range(len(candidates))}
        cand_scores = {i: [] for i in range(len(candidates))}
        cand_labels = {i: [] for i in range(len(candidates))}

        gkf = GroupKFold(n_splits=n_inner_splits)
        for inner_tr_idx, inner_val_idx in gkf.split(X_tr, y_tr, groups=g_tr):
            assert not (set(g_tr[inner_tr_idx]) & set(g_tr[inner_val_idx])), \
                f"inner leakage within outer fold {test_participant}"
            n_inner_checked += 1

            X_itr, y_itr = X_tr[inner_tr_idx], y_tr[inner_tr_idx]
            X_ival, y_ival = X_tr[inner_val_idx], y_tr[inner_val_idx]
            if len(np.unique(y_itr)) < 2 or len(np.unique(y_ival)) < 2:
                continue

            for i, (_, _, pipe) in enumerate(candidates):
                fitted = clone(pipe).fit(X_itr, y_itr)
                s = score_fn(fitted, X_ival)
                cand_aucs[i].append(roc_auc_score(y_ival, s))
                cand_scores[i].append(s)
                cand_labels[i].append(y_ival)

        mean_aucs = {i: np.mean(v) for i, v in cand_aucs.items() if v}
        best_i = max(mean_aucs, key=mean_aucs.get)
        best_name, best_c, best_template = candidates[best_i]

        pooled_inner_scores = np.concatenate(cand_scores[best_i])
        pooled_inner_labels = np.concatenate(cand_labels[best_i])
        threshold = best_threshold(pooled_inner_scores, pooled_inner_labels)

        final_pipe = clone(best_template).fit(X_tr, y_tr)
        test_scores = score_fn(final_pipe, X_te)
        oof_scores[outer_test_mask] = test_scores
        oof_preds[outer_test_mask] = (test_scores >= threshold).astype(int)

        fold_records.append({
            "test_participant": test_participant,
            "n_test": int(outer_test_mask.sum()),
            "chosen_model": best_name,
            "chosen_C": best_c,
            "inner_cv_auc": mean_aucs[best_i],
            "threshold": threshold,
        })

    print(f"Verified zero participant overlap: {n_outer_checked} outer folds, "
          f"{n_inner_checked} inner GroupKFold(6) splits.")
    return oof_scores, oof_preds, pd.DataFrame(fold_records)


def bootstrap_auc_ci(participant_ids, y_true, scores, n_boot=1000, seed=RANDOM_SEED):
    rng = np.random.default_rng(seed)
    participants = np.unique(participant_ids)
    idx_by_participant = {p: np.where(participant_ids == p)[0] for p in participants}
    boot_aucs = []
    for _ in range(n_boot):
        sampled = rng.choice(participants, size=len(participants), replace=True)
        idx = np.concatenate([idx_by_participant[p] for p in sampled])
        y_boot, s_boot = y_true[idx], scores[idx]
        if len(np.unique(y_boot)) < 2:
            continue
        boot_aucs.append(roc_auc_score(y_boot, s_boot))
    lo, hi = np.percentile(boot_aucs, [2.5, 97.5])
    return lo, hi, np.array(boot_aucs)

## 2. Reproduce the Real Result

Rerun the real pipeline on layer 12 before touching the permutation test — this should match Day 3's logged AUC 0.611 / CI [0.551, 0.683] almost exactly (small differences are possible: no random seed was fixed for the model fits in the original run either, but `GroupKFold` splits are deterministic given fixed group order, so any gap should be tiny). A large gap here means something changed and the permutation test below isn't testing the result we think it is.

In [ ]:
oof_scores, oof_preds, fold_info = nested_loso_cv(X, y, groups)
pooled_auc = roc_auc_score(y, oof_scores)
ci_lo, ci_hi, _ = bootstrap_auc_ci(groups, y, oof_scores)

print(f"Reproduced HuBERT layer {LAYER} pooled AUC: {pooled_auc:.3f}")
print(f"95% bootstrap CI: [{ci_lo:.3f}, {ci_hi:.3f}]")
print("(compare to Day 3's logged AUC 0.611, CI [0.551, 0.683] in PROJECT_PLAN.md)")
fold_info

## 3. Permutation Test

**Shuffling scheme:** labels are shuffled *within* each participant, across that participant's own sessions — preserving each participant's class balance and keeping all 3 activities of a session glued to the same (now-shuffled) label, exactly matching the real label structure. Same scheme used for the project's original baseline permutation test (`02_train_evaluate.ipynb` section 8).

**Compute shortcut, stated explicitly:** each permutation reuses the same `(model type, C)` already chosen per outer fold in the real run above (`fold_info`), rather than re-running the full inner hyperparameter search for all 1000 permutations — the permutation test asks whether there's *any* signal at all, not which regularization strength is best under fake labels.

In [ ]:
def shuffle_labels_within_participant(df, rng):
    y_perm = df["y"].to_numpy().copy()
    for p, group in df.groupby("participant_id"):
        session_labels = group.drop_duplicates("session_number")[["session_number", "y"]]
        shuffled = session_labels["y"].to_numpy().copy()
        rng.shuffle(shuffled)
        mapping = dict(zip(session_labels["session_number"], shuffled))
        y_perm[group.index] = group["session_number"].map(mapping).to_numpy()
    return y_perm


def run_fixed_hparams_loso(X, y, groups, fold_info):
    oof = np.zeros(len(y), dtype=float)
    for _, row in fold_info.iterrows():
        mask_test = groups == row["test_participant"]
        mask_train = ~mask_test
        if len(np.unique(y[mask_train])) < 2:
            continue
        pipe = build_pipeline(row["chosen_model"], row["chosen_C"])
        fitted = clone(pipe).fit(X[mask_train], y[mask_train])
        oof[mask_test] = score_fn(fitted, X[mask_test])
    return oof

In [ ]:
N_PERMUTATIONS = 1000  # this is the confirmatory run for the current leading candidate

rng = np.random.default_rng(RANDOM_SEED)
perm_aucs = []
for _ in tqdm(range(N_PERMUTATIONS), desc="Permutation test"):
    y_perm = shuffle_labels_within_participant(df, rng)
    if len(np.unique(y_perm)) < 2:
        continue
    perm_scores = run_fixed_hparams_loso(X, y_perm, groups, fold_info)
    perm_aucs.append(roc_auc_score(y_perm, perm_scores))

perm_aucs = np.array(perm_aucs)
p_value = (1 + np.sum(perm_aucs >= pooled_auc)) / (1 + len(perm_aucs))

print(f"Observed pooled AUC:  {pooled_auc:.3f}")
print(f"Null (permuted) mean: {perm_aucs.mean():.3f} +/- {perm_aucs.std():.3f}")
print(f"Empirical p-value:    {p_value:.4f}  (fraction of {len(perm_aucs)} permutations scoring >= observed)")

In [ ]:
SEQ = "#2a78d6"    # sequential single hue for the null distribution
OBS = "#eb6834"    # marker for the observed (real) AUC
INK, MUTED, GRID = "#0b0b0b", "#898781", "#e1e0d9"

fig, ax = plt.subplots(figsize=(8, 4.5))
ax.hist(perm_aucs, bins=30, color=SEQ, edgecolor="white", linewidth=0.5, zorder=3)
ax.axvline(pooled_auc, color=OBS, linewidth=2, zorder=4, label=f"Observed AUC = {pooled_auc:.3f}")
ax.axvline(0.5, color=MUTED, linewidth=1, linestyle="--", zorder=2)
ax.set_xlabel("Pooled ROC-AUC under label-shuffled (null) data", color=INK)
ax.set_ylabel(f"Count (out of {len(perm_aucs)} permutations)", color=INK)
ax.tick_params(colors=MUTED)
ax.spines[["top", "right"]].set_visible(False)
ax.grid(axis="y", color=GRID, linewidth=0.8, zorder=0)
ax.set_axisbelow(True)
ax.legend(frameon=False, labelcolor=INK)
plt.tight_layout()
plt.show()

## How to read this

- **p < 0.05** (fewer than ~5% of scrambled-label runs did as well as 0.611): treat HuBERT layer 12 as a confirmed real signal, not a fluke of this dataset's shape. It becomes the confirmed leading embedding for Day 6.
- **p in roughly 0.05-0.2**: suggestive but not conclusive — comparable to the project's original baseline permutation result (p=0.19), which this plan explicitly treated as "not significant." Report honestly as encouraging but unconfirmed, not a proven finding.
- **p > 0.2**: scrambled labels reach 0.611 too often for this to be trusted as real signal — despite the promising bootstrap CI, this result would need to be treated the same as every other Day 1-2 result: a directional lean, not a win.

Update `PROJECT_PLAN.md`'s Day 3 entry with this p-value once it's in, and factor it into whatever Day 4 (TRILLsson) and Day 6 decide.